# vla.cpp - run SmolVLA on LIBERO (Colab)

Runs the steps of [apps/vla.cpp/README.md](https://github.com/KuSenpai/modelopt101/blob/main/apps/vla.cpp/README.md) on a Colab GPU runtime:

1. build the `vla-server` with CUDA,
2. create the LIBERO simulation env for the client,
3. download the SmolVLA GGUF bundle,
4. start the server, run one LIBERO episode, and play the output video.

**Runtime:** `Runtime -> Change runtime type -> GPU`, then run the cells top to bottom.
The build and the LIBERO env each take roughly 10-20 minutes the first time.

Two differences from the README, because upstream `vla.cpp` changed:

- there is no `patches/patch.sh` any more - CMake now patches llama.cpp itself;
- the bundle is a single `smolvla-libero.gguf` (no `mmproj-*.gguf`), so the server takes one file.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5


In [ ]:
!ls -la /content/vla.cpp/build/vla-server /content/vla.cpp/eval/sim/libero; env | grep ^UV_

-rwxr-xr-x 1 root root 212672 Nov  3 04:27 /content/vla.cpp/build/vla-server

/content/vla.cpp/eval/sim/libero:
total 40
drwxr-xr-x 4 root root  4096 Nov  3 04:34 .
drwxr-xr-x 4 root root  4096 Nov  3 04:13 ..
-rw-r--r-- 1 root root   867 Nov  3 04:13 __init__.py
drwxr-xr-x 9 root root  4096 Nov  3 04:34 LIBERO
-rw-r--r-- 1 root root 15672 Nov  3 04:13 libero_env.py
drwxr-xr-x 3 root root  4096 Nov  3 04:34 libero_uv
-rw-r--r-- 1 root root  2913 Nov  3 04:13 setup_libero.sh
UV_BUILD_CONSTRAINT=
UV_SYSTEM_PYTHON=true
UV_INSTALL_DIR=/usr/local/bin
UV_BREAK_SYSTEM_PACKAGES=false
UV_CONSTRAINT=


In [ ]:
%%bash
cd /content/vla.cpp
for v in $(env | grep -o '^UV_[A-Z_]*' || true); do unset "$v"; done
nohup bash eval/sim/libero/setup_libero.sh > /content/libero_setup.log 2>&1 &
echo started

started


In [ ]:
!tail -n 15 /content/libero_setup.log; pgrep -f setup_libero.sh > /dev/null && echo RUNNING || echo FINISHED

Uninstalled 1 package in 7ms
Installed 1 package in 1ms
 - mujoco==3.17.0
 + mujoco==2.3.2
+ rm -rf /root/.libero
+ echo N
+ MUJOCO_GL=egl
+ python -c 'import libero.libero'
Do you want to specify a custom path for the dataset folder? (Y/N): Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero
bddl_files: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/./bddl_files
init_states: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/./init_files
datasets: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/../datasets
assets: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/./assets
RUNNING


## 1. Build the server

`vla.cpp`'s CUDA build needs an **Ampere or newer GPU (`sm_80`+)**: its FoldQuant kernels use
`mma.m16n8k64` and `cp.async`, which older GPUs lack, and CMake always builds them. So:

- **L4 / A100** (Colab Pro): CUDA build, as in the README.
- **T4** (free Colab, `sm_75`): the CUDA build fails in `ptxas`, so this cell builds the
  **CPU** server instead. It works, just slower per action.

In [ ]:
%%bash
set -euxo pipefail
apt-get -qq update
apt-get -qq install -y libzmq3-dev cppzmq-dev libprotobuf-dev protobuf-compiler > /dev/null

cd /content
[ -d vla.cpp ] || git clone --depth 1 https://github.com/VinRobotics/vla.cpp.git
cd vla.cpp

export PATH=/usr/local/cuda/bin:$PATH
CUDA_ARCHITECTURE=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
if [ "$CUDA_ARCHITECTURE" -ge 80 ]; then
    CUDA=ON
else
    CUDA=OFF
    echo "sm_$CUDA_ARCHITECTURE < sm_80: vla.cpp CUDA kernels need Ampere+, building the CPU server"
fi
# a build dir configured for the other backend (e.g. a failed CUDA try) is reconfigured from scratch
grep -qs "GGML_CUDA:BOOL=$CUDA" build/CMakeCache.txt || rm -rf build
cmake -B build \
    -DGGML_CUDA=$CUDA \
    -DGGML_CUDA_GRAPHS=ON \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES=$CUDA_ARCHITECTURE
cmake --build build -j$(nproc)
ls -lh build/vla-server

sm_75 < sm_80: vla.cpp CUDA kernels need Ampere+, building the CPU server
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.5.0-dev
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCES

+ apt-get -qq update
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
+ apt-get -qq install -y libzmq3-dev cppzmq-dev libprotobuf-dev protobuf-compiler
+ cd /content
+ '[' -d vla.cpp ']'
+ git clone --depth 1 https://github.com/VinRobotics/vla.cpp.git
Cloning into 'vla.cpp'...
+ cd vla.cpp
+ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
+ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ tr -d .
++ nvidia-smi --query-gpu=compute_cap --format=csv,noheader
++ head -1
+ CUDA_ARCHITECTURE=75
+ '[' 75 -ge 80 ']'
+ CUDA=OFF
+ echo 'sm_75 < sm_80: vla.cpp CUDA kernels need Ampere+, bu

## 2. Create the LIBERO simulation env with uv

`setup_libero.sh` clones LIBERO and builds a Python 3.10 venv in `eval/sim/libero/libero_uv/.venv`.
MuJoCo renders headless through EGL on the GPU; the snippet registers NVIDIA's EGL driver,
which Colab images do not always ship a config for.

In [ ]:
%%bash
set -euxo pipefail
pip -q install uv
for v in $(env | grep -o '^UV_[A-Z_]*' || true); do unset "$v"; done
ICD=/usr/share/glvnd/egl_vendor.d/10_nvidia.json
if [ ! -f "$ICD" ]; then
  mkdir -p "$(dirname "$ICD")"
  echo '{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}' > "$ICD"
fi
cd /content/vla.cpp
bash eval/sim/libero/setup_libero.sh

Do you want to specify a custom path for the dataset folder? (Y/N): Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero
bddl_files: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/./bddl_files
init_states: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/./init_files
datasets: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/../datasets
assets: /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/./assets


+ pip -q install uv
++ env
++ grep -o '^UV_[A-Z_]*'
+ for v in $(env | grep -o '^UV_[A-Z_]*' || true)
+ unset UV_BUILD_CONSTRAINT
+ for v in $(env | grep -o '^UV_[A-Z_]*' || true)
+ unset UV_SYSTEM_PYTHON
+ for v in $(env | grep -o '^UV_[A-Z_]*' || true)
+ unset UV_INSTALL_DIR
+ for v in $(env | grep -o '^UV_[A-Z_]*' || true)
+ unset UV_BREAK_SYSTEM_PACKAGES
+ for v in $(env | grep -o '^UV_[A-Z_]*' || true)
+ unset UV_CONSTRAINT
+ ICD=/usr/share/glvnd/egl_vendor.d/10_nvidia.json
+ '[' '!' -f /usr/share/glvnd/egl_vendor.d/10_nvidia.json ']'
++ dirname /usr/share/glvnd/egl_vendor.d/10_nvidia.json
+ mkdir -p /usr/share/glvnd/egl_vendor.d
+ echo '{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}'
+ cd /content/vla.cpp
+ bash eval/sim/libero/setup_libero.sh
+ export CMAKE_POLICY_VERSION_MINIMUM=3.5
+ CMAKE_POLICY_VERSION_MINIMUM=3.5
+++ dirname eval/sim/libero/setup_libero.sh
++ cd eval/sim/libero
++ pwd
+ SCRIPT_DIR=/content/vla.cpp/eval/sim/libero
+ LIBERO_RE

## 3. Get the SmolVLA bundle

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("vrfai/smolvla-libero-gguf", local_dir="/content/vla.cpp/models/smolvla")
!ls -lh /content/vla.cpp/models/smolvla

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

total 1.1G
-rw-r--r-- 1 root root 2.0K Nov  4 02:13 README.md
-rw-r--r-- 1 root root 1.1G Nov  4 02:14 smolvla-libero.gguf


## 4. Run

Start the server in the background and wait until it reports it is bound and ready.
The `vla: backend = ...` line says where it runs (`CUDA` on L4/A100, `CPU` on a T4).

In [ ]:
import re, subprocess, time

REPO = "/content/vla.cpp"
LOG = "/content/vla-server.log"

server = subprocess.Popen(
    ["./build/vla-server", "--bind", "tcp://*:5555", "models/smolvla/smolvla-libero.gguf"],
    cwd=REPO, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
)
for _ in range(600):
    log = open(LOG).read()
    if re.search(r"bound to .* ready", log):
        break
    if server.poll() is not None:
        raise RuntimeError("vla-server exited:\n" + log[-3000:])
    time.sleep(1)
else:
    raise TimeoutError("vla-server not ready after 600s:\n" + log[-3000:])

print(log[-2000:])
print([l for l in log.splitlines() if "backend =" in l])

vla-server: loading model ...
  ckpt:   models/smolvla/smolvla-libero.gguf
vla: arch = smolvla
vla: config = models/smolvla/smolvla-libero.gguf (gguf KV)
vla: backend = CPU (2 threads)
vla: [vram] after backend init     used=0.0 MiB  free=12975.5 MiB
vla: tower weights resident as bf16
vla: cfg  hidden=960 expert_h=480 expert_inter=1280 n_layers=32 n_img=64 n_lang=48 n_suffix=50 max_state=32 max_action=32
vla: cfg  real_state_dim=8 real_action_dim=7
vla: [vram] weight_buf = 985.0 MiB
vla: [vram] after weights alloc    used=0.0 MiB  free=12975.5 MiB  (+0.0 MiB)
vla-server: loaded. chunk_size=50  action_dim=32  n_lang=48  hidden=960  expert_h=480  timing_detail=none
vla(smolvla): image view is 224x224, expected 512x512
vla-server: warmup predict failed (synthetic inputs); continuing
vla-server: bound to tcp://*:5555. ready.
vla-server: WARNING: bound to tcp://*:5555 with no authentication. Any host that
            can reach this address may submit requests. Restrict access to a
        

Run one LIBERO episode (the README's client command, using the venv's Python directly
because `source .venv/bin/activate` does not carry across notebook cells).

`--n-action-steps 10` executes 10 actions of each predicted chunk before asking the server
again - 10x fewer model calls, which matters on the CPU server. It is also what upstream's
LIBERO sweep uses for SmolVLA (the SmolVLA paper found 10 beats 1).

In [ ]:
%%bash
set -euxo pipefail
cd /content/vla.cpp
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
eval/sim/libero/libero_uv/.venv/bin/python eval/client/run_sim_client_direct.py \
    --task libero_object --task-id 0 --n-episodes 1 \
    --output-dir /tmp/libero_outputs \
    --arch smolvla --n-action-steps 10

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
vla-cpp-direct[arch=smolvla]: connected to tcp://localhost:5555
vla-cpp-direct: loading tokenizer HuggingFaceTB/SmolVLM2-500M-Instruct
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[Warning]: datasets path /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/../datasets does not exist!
[Warning]: datasets path /content/vla.cpp/eval/sim/libero/LIBERO/libero/libero/../datasets does not exist!
*** Episode 1/1
- Episode finished after 153 steps.
- Final reward: 1.00
- Episode Information:
{'task': 'pick_up_the_alphabet_soup_and_place_it_in_the_basket', 'task_id': 0, 'done': True, 'is_success': True, 'final_info': {'task': 'pick_up_the_alphabet_soup_and_place_it_in_the_basket', 'task_id': 0, 'done': True, 'is_success': True}}
- Average inference time per step:

+ cd /content/vla.cpp
+ export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
+ MUJOCO_GL=egl
+ PYOPENGL_PLATFORM=egl
+ eval/sim/libero/libero_uv/.venv/bin/python eval/client/run_sim_client_direct.py --task libero_object --task-id 0 --n-episodes 1 --output-dir /tmp/libero_outputs --arch smolvla --n-action-steps 10
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /content/vla.cpp/eval/sim/libero/libero_uv/.venv/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/content/vla.cp

## 5. Result

The client writes the video, `summary.txt` and `episodes.csv` to
`/tmp/libero_outputs/smolvla/libero_object/task_0/`.

In [ ]:
import base64, glob
from IPython.display import HTML, display

out = "/tmp/libero_outputs/smolvla/libero_object/task_0"
print(open(f"{out}/summary.txt").read())
video = sorted(glob.glob(f"{out}/*.mp4"))[-1]
b64 = base64.b64encode(open(video, "rb").read()).decode()
display(HTML(f'<video width="640" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

Arch: smolvla
Task: libero_object/task_0
n_action_steps: 10
Success rate: 100.00%  (1/1)
Skipped (terminated mid-step): 0/1
Average inference time per step: 1721.89 ms

